# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Classification (binary).** The question is "will this page decline next half — yes or no?",
which is exactly the `framing-ml-problems/SKILL.md` "will this one decline / recover?" row —
target is a yes/no label, success metric is precision/recall against a base rate.

It's not ranking/scoring in the pure sense, even though the end product (the reviewed queue) is
ordered — the model outputs a probability per (client, content) pair, and the queue is that
probability sorted descending. Classification underneath, ranking on top for the deliverable.


In [5]:
# Task-type sanity check: the target is binary, not continuous or multi-class.
print("Target: is_declining_next_half")
print("Values it can take: {0, 1}")
print("This matches classification, not regression (no continuous target) or clustering (target exists).")


Target: is_declining_next_half
Values it can take: {0, 1}
This matches classification, not regression (no continuous target) or clustering (target exists).


## 2. Target or proxy

**Target:** `is_declining_next_half` — 1 if a (client, content) pair's summed GSC impressions in
Mar 16-31 (the label window) are more than 20% lower than Mar 1-15 (the feature window), else 0.

**This is an observed outcome, not a defined rule pretending to be one.** The 20% threshold is a
definitional choice (it mirrors the starter dataset's own `trend_direction == "down"` rule), but
what it's measuring — whether impressions actually dropped in a real, strictly-later time window —
already happened in the data by the time I compute it. The model never sees Mar 16-31 as a
feature; it only sees the label built from it, kept in a completely separate bucket
(`w03_data_contract.ipynb`'s field-bucket table: feature / label / context / excluded).


In [6]:
# Reproduces the base-rate check already committed in w03/w04: confirms the target is neither
# near-0% nor near-100%, so it's learnable and non-trivial.
print("Base rate of is_declining_next_half (from committed w04_baseline_score.ipynb run): 32.7%")
print("Total (client, content) pairs in the modeling population: 151,981")
print("This sits comfortably away from both 0% and 100% -- a real, learnable target.")


Base rate of is_declining_next_half (from committed w04_baseline_score.ipynb run): 32.7%
Total (client, content) pairs in the modeling population: 151,981
This sits comfortably away from both 0% and 100% -- a real, learnable target.


## 3. Success metric

**Primary: Precision@10 on held-out (never-trained-on) clients.** Given the cost asymmetry from
`w01` (a missed decline is expensive, a wasted review is cheap), what matters most is: *can the
very top of the ranked list be trusted?* Precision@10 answers that directly — of the 10 pairs the
model is most confident about, how many actually declined?

**Secondary: Precision@50 and ROC-AUC**, for a fuller picture beyond the top 10 — Precision@50
checks whether the list stays trustworthy as the review budget grows, and ROC-AUC checks overall
ranking quality independent of any one cutoff.

**The floor to beat:** the rule-based baseline already scores **Precision@10 = 50.0%** against a
**32.7% base rate**, on the whole month (not yet held-out clients) — that's the number `w05_model`
has to beat on a fair, client-held-out comparison to justify using a model over the plain rule.


In [7]:
def precision_at_k(y_true, scores, k):
    import numpy as np
    order = np.argsort(-scores)
    top_k = np.asarray(y_true)[order][:k]
    return top_k.mean()

print("Metric: Precision@K")
print("Baseline rule (w04_baseline_score.ipynb, whole month, not client-held-out):")
print("  base_rate = 32.7%")
print("  precision_at_10 = 50.0%")
print("\nThis notebook's job is to define the metric; w05_model.ipynb runs the fair,")
print("client-held-out comparison against this floor.")


Metric: Precision@K
Baseline rule (w04_baseline_score.ipynb, whole month, not client-held-out):
  base_rate = 32.7%
  precision_at_10 = 50.0%

This notebook's job is to define the metric; w05_model.ipynb runs the fair,
client-held-out comparison against this floor.


## 4. The unit of analysis, as a real dataframe

**One row = one (client, content) pair's activity within March 2026**, split into a first-half
feature window (Mar 1-15) and a second-half label window (Mar 16-31), built from
`fact_content_daily_performance`.

Grain confirmed in `w03_data_contract.ipynb`: 0 duplicate `(report_date, client_hash_id,
content_hash_id)` combinations in the raw daily table, and after aggregating to the
first-half/second-half pair level, 151,981 unique (client, content) rows with data present in
both halves.


In [8]:
# ---- Same setup/query pattern as w01, w03, w04, w05 -- run in Colab with your own HF_TOKEN ----
%pip -q install duckdb
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN (plain Read token): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"

unit_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING c > 1
""").df()
print(f"Duplicate (date, client, content) rows: {len(unit_check)}  -- expect 0, per w03's committed run")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) rows: 0  -- expect 0, per w03's committed run


## 5. Why ML beats a fixed rule here

Because a fixed rule was already tried, on the same data, and it's mediocre in a specific,
diagnosable way. `w04_baseline_score.ipynb`'s rule combines exactly two signals by hand — has
enough volume, and CTR below what its own position tier typically gets — and both signal checks
came back **"MIXED"** (not cleanly confirmed) rather than a clean monotonic pattern. The rule
still beats the base rate (50.0% vs 32.7%), but "mixed" signals are exactly the case
`framing-ml-problems/SKILL.md` describes as too messy for an if-statement: the pattern is real
(the rule beats chance) but tangled — volume, position, CTR-vs-expectation, and day-coverage
likely interact (e.g. low CTR only matters *combined with* enough active days to trust it), which
is precisely what Logistic Regression and Random Forest can weigh and combine that a hand-written
AND/OR rule can't.


In [9]:
print("Signal verdicts from w04_baseline_score.ipynb (committed run):")
print("  signal1_ctr_vs_position_verdict:", "MIXED")
print("  signal2_volume_vs_decline_verdict:", "MIXED")
print()
print("Baseline still beats the base rate (50.0% vs 32.7% precision@10),")
print("but 'MIXED' on both individual signals is the concrete evidence that a single")
print("hand-written AND/OR rule isn't cleanly separating the pattern -- motivating w05_model.")


Signal verdicts from w04_baseline_score.ipynb (committed run):
  signal1_ctr_vs_position_verdict: MIXED
  signal2_volume_vs_decline_verdict: MIXED

Baseline still beats the base rate (50.0% vs 32.7% precision@10),
but 'MIXED' on both individual signals is the concrete evidence that a single
hand-written AND/OR rule isn't cleanly separating the pattern -- motivating w05_model.
